Celda 1: El Cerebro (Librerías y Constantes)

In [ ]:
import os
import re
import pdfplumber
import pandas as pd

MUROS_DIVISORES = {'BANDEJA', 'KILOGRAMO', 'JABA', 'CAJON', 'SACO', 'ATADO', 'UNIDAD', 'KG', 'BALDE', 'CAJA', 'BOLSA', 'CIENTO', 'DOCENA', 'PAQUETE', 'MALLA'}

MESES_MAP = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04','mayo':'05','junio':'06',
    'julio':'07','agosto':'08','septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

CATALOGO_OFICIAL_FRUTAS = {
    'AGUAYMANTO', 'ALBARICOQUE', 'ARÁNDANO', 'CAMU CAMU', 'CARAMBOLA',
    'CHIRIMOYA CRIOLLA O HUAYCO', 'CHIRIMOYA CUMBE', 'CIRUELA SANTA ROSA/NACIONAL',
    'COCO', 'COCONA', 'DATIL MELADO', 'FRAMBUESA', 'FRESA SABRINA', 'FRESA SAN ANDREAS',
    'GRANADA', 'GRANADILLA (COSTA)', 'GRANADILLA (SELVA)', 'GUANABANA (COSTA)',
    'GUANABANA (SELVA)', 'GUAYABA', 'HIGO MADURO', 'KAKE/KAKI', 'KIWI IMPORTADO',
    'LIMA DULCE', 'LUCUMA BELTRAN', 'LUCUMA DE SEDA', 'MAMEY', 'MANDARINA CLEMENTINA',
    'MANDARINA MURCOTT', 'MANDARINA PRIMOSOLE', 'MANDARINA SATSUMA', 'MANDARINA TANGERINA',
    'MANGO CRIOLLO', 'MANGO EDWARD', 'MANGO HADEN /HAYDE', 'MANGO KAFRO', 'MANGO KENT',
    'MANZANA CTE/PARA AGUA', 'MANZANA DELICIA', 'MANZANA GOLDEN', 'MANZANA IMPORTADA',
    'MANZANA ISRAEL', 'MANZANA WINTER', 'MARACUYA', 'MELOCOTON BLANQUILLO',
    'MELOCOTON DURAZNO HUAYCO', 'MELON COQUITO', 'MELON MENTA', 'MEMBRILLO CRIOLLO',
    'MEMBRILLO SERRANO', 'NARANJA PRIMAVERA', 'NARANJA TANGELO SELVA', 'NARANJA VALENCIA',
    'NARANJA WASHINGTON NAVAL', 'NARANJITA CHINA KIN KAN', 'NISPERO DE AGUA', 'NONI',
    'PACAE CRIOLLO', 'PACAE DE SEDA', 'PALTA CHOQUETE', 'PALTA COLLIN RED', 'PALTA CRIOLLA',
    'PALTA DEDO', 'PALTA ETTINGER', 'PALTA FUERTE', 'PALTA HALL', 'PALTA HASS', 'PALTA LINDA',
    'PALTA NAVAL', 'PALTA ZUTANO', 'PAPAYA', 'PEPINO RAYADO O MELON',
    'PERA AGUA PACKAM IMPORTADO', 'PITAHAYA', 'PIÑA CRIOLLA', 'PIÑA GOLDEN', 'PIÑA HAWAIANA',
    'PLATANO BELLACO', 'PLATANO BIZCOCHO', 'PLATANO ISLA', 'PLATANO MANZANO', 'PLATANO MORADO',
    'PLATANO PALILLO', 'PLATANO SEDA (SELVA)', 'PLATANO SEDA CONGO', 'SANDIA', 'SANKI',
    'TAMARINDO (CON CASCARA)', 'TORONJA COSTA', 'TORONJA SELVA', 'TUMBO', 'TUNA AMARILLA',
    'TUNA BLANCA', 'TUNA MORADA/ROSADA', 'UVA ALFONSO LAVALETT', 'UVA ARRA', 'UVA BORGOÑA',
    'UVA ITALIA', 'UVA PALESTINA', 'UVA PIROVANO', 'UVA RED GLOBE', 'UVA ROSADA',
    'UVA SUPERIOR', 'UVA THOMPSON', 'YACON (LLACON)', 'ZARZAMORA/MORA'
}

def format_fecha(txt):
    try:
        txt = txt.lower()
        for m_es, m_num in MESES_MAP.items():
            if f' de {m_es} de ' in txt: txt = txt.replace(f' de {m_es} de ', f'/{m_num}/')
            elif f' de {m_es} del ' in txt: txt = txt.replace(f' de {m_es} del ', f'/{m_num}/')
        return txt
    except:
        return None

print("Celda 1: Dependencias, constantes y Catálogo Oficial cargados correctamente.")

Celda 2: El Motor de Extracción (Solo define la función)

In [ ]:
def procesar_pdf_diario(ruta_pdf):
    if not os.path.exists(ruta_pdf):
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_pdf}")

    resultados = []
    with pdfplumber.open(ruta_pdf) as pdf:
        texto_pag1 = pdf.pages[0].extract_text()
        match_fecha = re.search(r'(\d{1,2}\s+de\s+\w+\s+de\s+(\d{4}))', texto_pag1, re.IGNORECASE)
        fecha_str = match_fecha.group(1) if match_fecha else os.path.basename(ruta_pdf).replace('.pdf', '')

        for pagina in pdf.pages:
            tabla = pagina.extract_table()
            if not tabla: continue

            for fila in tabla:
                if not fila or fila[0] is None: continue

                # 1. Limpieza inicial del nombre del producto
                prod_raw = str(fila[0]).split('\n')[0]
                nombre_item = ' '.join(re.sub(r'[\d\.]+', '', prod_raw).split()).upper().strip()

                # --- 2. ACTIVACIÓN DEL ESCUDO (FILTRO) ---
                # Saltar filas vacías o cabeceras
                if not nombre_item or nombre_item in ['PRODUCTOS', 'NONE', 'PRECIOS', 'VARIACIÓN', '']:
                    continue

                # Saltar vegetales, notas de pie de página o frutas mal escritas
                if nombre_item not in CATALOGO_OFICIAL_FRUTAS:
                    continue
                # -----------------------------------------

                # 3. Búsqueda del "muro divisor" para separar texto de números
                idx_muro = -1
                for idx_c, celda in enumerate(fila):
                    if celda:
                        celda_upper = str(celda).upper().strip().replace('\n', '')
                        if any(muro in celda_upper for muro in MUROS_DIVISORES):
                            idx_muro = idx_c; break
                if idx_muro == -1: continue

                # 4. Extracción de volúmenes y precios
                texto_izq = ' '.join([str(fila[i]) for i in range(1, idx_muro) if fila[i]])
                vols = re.findall(r'\d+\.\d+|\d+', texto_izq)

                texto_der = ' '.join([str(fila[i]) for i in range(idx_muro + 2, len(fila)) if fila[i]])
                precios = re.findall(r'\d+\.\d+|\d+', texto_der.replace(',', '.'))

                # 5. Guardado temporal si los datos están completos
                if len(vols) >= 1 and len(precios) >= 2:
                    resultados.append({
                        'fecha_original': fecha_str, 'producto': nombre_item,
                        'volumen_hoy': vols[0], 'volumen_7dias': vols[1] if len(vols) > 1 else vols[0],
                        'precio_ayer': precios[0], 'precio_7dias': precios[2] if len(precios) > 2 else precios[0],
                        'precio_hoy_kg': precios[1] if len(precios) > 1 else precios[0]
                    })

    if not resultados: return pd.DataFrame()

    # Formateo y ordenamiento final
    df_diario = pd.DataFrame(resultados)
    for col in ['volumen_hoy', 'volumen_7dias', 'precio_ayer', 'precio_7dias', 'precio_hoy_kg']:
        df_diario[col] = pd.to_numeric(df_diario[col], errors='coerce').fillna(0).round(2)

    df_diario['fecha_dt'] = pd.to_datetime(df_diario['fecha_original'].apply(format_fecha), errors='coerce', dayfirst=True)
    df_diario = df_diario.dropna(subset=['fecha_dt']).sort_values(['fecha_dt', 'producto'])

    return df_diario[[
        'fecha_dt', 'producto', 'volumen_hoy', 'volumen_7dias', 'precio_ayer', 'precio_7dias', 'precio_hoy_kg'
    ]].rename(columns={
        'fecha_dt': 'Fecha', 'producto': 'Producto', 'volumen_hoy': 'Volumen_Hoy',
        'volumen_7dias': 'Volumen_7Dias', 'precio_ayer': 'Precio_Ayer',
        'precio_7dias': 'Precio_7Dias', 'precio_hoy_kg': 'Precio_Hoy_Kg'
    })

print("Celda 2: Motor de extracción PDF compilado con Aduana de Datos Activa.")

Celda 3: La Zona de Pruebas (El "Freno de Emergencia")

In [ ]:
# Asegúrate de subir tu PDF de prueba a Colab antes de correr esto
archivo_prueba = 'reporte_hoy.pdf' # Cambia esto por el nombre real de tu PDF

print(f"Extrayendo datos de {archivo_prueba}...")
df_nuevo_dia = procesar_pdf_diario(archivo_prueba)

# Vemos los primeros 10 resultados para confirmar que todo está perfecto
display(df_nuevo_dia.head(10))
print(f"Total de frutas extraídas: {len(df_nuevo_dia)}")

Celda 4: Inyección en la Base de Datos Histórica

In [ ]:
def actualizar_historico(df_nuevo, ruta_csv_maestro='dataset_productos_anual.csv'):
    if df_nuevo.empty:
        print("Cancelado: No hay datos para actualizar.")
        return

    if os.path.exists(ruta_csv_maestro):
        df_historico = pd.read_csv(ruta_csv_maestro)
        df_historico['Fecha'] = pd.to_datetime(df_historico['Fecha'])
    else:
        df_historico = pd.DataFrame()

    df_actualizado = pd.concat([df_historico, df_nuevo], ignore_index=True)
    df_actualizado = df_actualizado.drop_duplicates(subset=['Fecha', 'Producto'], keep='last')
    df_actualizado = df_actualizado.sort_values(['Fecha', 'Producto'])
    df_actualizado.to_csv(ruta_csv_maestro, index=False)

    print(f"✅ ¡Éxito! Se añadieron {len(df_nuevo)} registros al maestro.")
    print(f"📊 Total de registros ahora en {ruta_csv_maestro}: {len(df_actualizado)}")

# EJECUTAR LA ACTUALIZACIÓN
actualizar_historico(df_nuevo_dia, 'dataset_productos_anual.csv')